# Notebook C1: Chạy MAX-SAT trên D-Wave (Hình 4 trên máy thật)

Đi kèm [docs/C-hardware/dwave-leap.md](../../docs/C-hardware/dwave-leap.md) và [Phần A, Mục 5](../../docs/A-translation/05-quantum-annealer.md).

**Notebook chạy được dù bạn có hay không có tài khoản D-Wave.** Ô cấu hình bên dưới thử kết nối tới máy QPU thật; nếu không có API token (hoặc kết nối thất bại), nó tự chuyển sang `SimulatedAnnealingSampler` chạy trên máy bạn. Mọi ô phụ thuộc vào QPU đều được ghi chú rõ và tự bỏ qua trong chế độ mô phỏng.

In [1]:
import itertools, os, warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import dimod, minorminer
import dwave_networkx as dnx
from dimod import Binary, SimulatedAnnealingSampler
from dwave.system import DWaveSampler, EmbeddingComposite

# Đặt USE_QPU=0 trong môi trường nếu muốn ép dùng trình mô phỏng.
USE_QPU = os.environ.get("USE_QPU", "1") == "1"
try:
    if not USE_QPU:
        raise RuntimeError("USE_QPU=0")
    qpu = DWaveSampler()                       # đọc token từ file cấu hình (dwave config create) hoặc biến DWAVE_API_TOKEN
    sampler = EmbeddingComposite(qpu)
    ON_QPU = True
    print("Đang dùng QPU:", qpu.solver.name, "| topology:", qpu.properties["topology"]["type"])
except Exception as e:
    sampler = SimulatedAnnealingSampler()
    ON_QPU = False
    print(f"Không dùng được QPU ({type(e).__name__}: {str(e)[:80]}).")
    print("-> Chuyển sang SimulatedAnnealingSampler (chạy cục bộ, không cần tài khoản).")

Không dùng được QPU (ValueError: API token not defined).
-> Chuyển sang SimulatedAnnealingSampler (chạy cục bộ, không cần tài khoản).


## 1. Mô hình: phương trình (39) với $P = 7$ (bản đã sửa)

Đây là QUBO đúng của Ví dụ 15, có đủ số hạng $-2x_3 z$. (Hình 4 trong bài gốc thiếu số hạng này; xem ghi chú ở Mục 5 của Phần A.)

In [2]:
x1, x2, x3, z = (Binary(v) for v in ("x1", "x2", "x3", "z"))
P = 7
model = x1*z - 2*z - x1*x3 + x2 + x3 + P*(x2*x3 - 2*x2*z - 2*x3*z + 3*z)

def count_unsat(a, b, c):
    clauses = [(a or (not b) or c), (a or b or (not c)), ((not a) or (not b) or c)]
    return sum(1 for cl in clauses if not cl)
satisfying = [t for t in itertools.product((0, 1), repeat=3) if count_unsat(*t) == 0]
print("Các phép gán thỏa mọi clause (đáp án đúng):", satisfying)

Các phép gán thỏa mọi clause (đáp án đúng): [(0, 0, 0), (0, 1, 1), (1, 0, 0), (1, 0, 1), (1, 1, 1)]


## 2. Minor-embedding: bài toán trông thế nào trên từng topology?

Máy D-Wave không cho phép coupling giữa hai qubit bất kỳ (Mục 5.2). Ta có thể thấy điều này **mà không cần máy thật**: dùng `minorminer` để nhúng đồ thị của QUBO lên các topology Chimera, Pegasus và Zephyr (bản nhỏ), rồi xem mỗi biến logic cần bao nhiêu qubit vật lý (độ dài chain).

In [3]:
bqm = dimod.BinaryQuadraticModel(model)
logical_edges = list(bqm.quadratic)
print("Các cạnh của đồ thị bài toán:", logical_edges)
print()
for name, G in (("Chimera (C2)", dnx.chimera_graph(2)),
                ("Pegasus (P2)", dnx.pegasus_graph(2)),
                ("Zephyr (Z1)", dnx.zephyr_graph(1))):
    emb = minorminer.find_embedding(logical_edges, G.edges, random_seed=1)
    chains = {v: len(q) for v, q in sorted(emb.items())}
    max_deg = max(d for _, d in G.degree())
    print(f"{name:13s} bậc tối đa mỗi qubit = {max_deg:2d} | độ dài chain của từng biến: {chains}")

Các cạnh của đồ thị bài toán: [('z', 'x1'), ('x3', 'x1'), ('x3', 'z'), ('x2', 'z'), ('x2', 'x3')]

Chimera (C2)  bậc tối đa mỗi qubit =  5 | độ dài chain của từng biến: {'x1': 1, 'x2': 1, 'x3': 2, 'z': 1}
Pegasus (P2)  bậc tối đa mỗi qubit = 13 | độ dài chain của từng biến: {'x1': 1, 'x2': 1, 'x3': 1, 'z': 1}
Zephyr (Z1)   bậc tối đa mỗi qubit = 17 | độ dài chain của từng biến: {'x1': 1, 'x2': 1, 'x3': 1, 'z': 1}


Chimera mỗi qubit chỉ nối tối đa 5 qubit khác, nên biến $x_3$ (nối tới 3 biến khác trong đồ thị bài toán, trong đó có cặp khó đặt cạnh nhau) phải dùng một chain 2 qubit. Pegasus và Zephyr có độ kết nối cao hơn nên bài toán nhỏ này nhúng được mà không cần chain. Chain càng dài thì càng dễ bị "đứt" (qubit trong chain nhận giá trị khác nhau), như đã nói ở Mục 5.2.

## 3. Chạy và đọc kết quả

Giống Hình 4 của bài báo: 1000 lần chạy (`num_reads=1000`). Cột `unsat` là số clause không được thỏa, `occ` là số lần xuất hiện.

In [4]:
kwargs = dict(num_reads=1000)
if ON_QPU:
    kwargs["label"] = "co-quantum maxsat ex15"      # nhãn để tìm lại job trên Leap dashboard
sampleset = sampler.sample(model, **kwargs)

agg = sampleset.aggregate()
print("  x1 x2 x3 z  energy  unsat  occ")
for rec in sorted(agg.data(["sample", "energy", "num_occurrences"]), key=lambda r: (r.energy, -r.num_occurrences)):
    s = rec.sample
    a, b, c, d = (int(s[v]) for v in ("x1", "x2", "x3", "z"))
    print(f"   {a}  {b}  {c}  {d}  {rec.energy:6.1f}  {count_unsat(a, b, c):5d}  {rec.num_occurrences:4d}")

found = {(int(r.sample["x1"]), int(r.sample["x2"]), int(r.sample["x3"]))
         for r in agg.data(["sample", "energy"]) if r.energy == 0}
print("\nCác phép gán Energy = 0 tìm được:", sorted(found))
print("Đủ cả 5 phép gán thỏa mọi clause?", found == set(satisfying))

  x1 x2 x3 z  energy  unsat  occ
   0  0  0  0     0.0      0   244
   1  0  1  0     0.0      0   240
   1  0  0  0     0.0      0   218
   1  1  1  1     0.0      0   156
   0  1  1  1     0.0      0   142

Các phép gán Energy = 0 tìm được: [(0, 0, 0), (0, 1, 1), (1, 0, 0), (1, 0, 1), (1, 1, 1)]
Đủ cả 5 phép gán thỏa mọi clause? True


Với QUBO đã sửa, ta kỳ vọng thấy **cả 5** phép gán `000`, `011`, `100`, `101`, `111` có Energy bằng 0 (khác với Bảng 2 của bài gốc, vốn chỉ có 3). Trên máy thật, vài trạng thái Energy 1 hoặc cao hơn cũng xuất hiện do nhiễu và do chain bị đứt; số lần xuất hiện cụ thể thay đổi giữa các lần chạy.

In [5]:
if ON_QPU:
    names = sampleset.record.dtype.names
    if "chain_break_fraction" in names:
        print("Tỉ lệ chain bị đứt trung bình:", float(sampleset.record.chain_break_fraction.mean()))
    timing = sampleset.info.get("timing", {})
    print("Thời gian QPU tính phí (qpu_access_time, micro giây):", timing.get("qpu_access_time"))
    print("Embedding dùng:", sampleset.info.get("embedding_context", {}).get("embedding"))
else:
    print("(Chế độ mô phỏng: không có thông tin chain hay thời gian QPU.)")

(Chế độ mô phỏng: không có thông tin chain hay thời gian QPU.)


## 4. Ảnh hưởng của `chain_strength` (chỉ chạy khi có QPU)

Coupling trong chain phải đủ mạnh để các qubit của chain đồng ý với nhau, nhưng quá mạnh thì "lấn át" phần bài toán thật. Ô dưới đây quét vài giá trị và đo tỉ lệ chain đứt cùng tỉ lệ lần chạy rơi vào một phép gán đúng. **Ô này tiêu tốn thời gian QPU** (mỗi giá trị 200 lần chạy), nên chỉ chạy khi `ON_QPU` là đúng.

In [6]:
def correct_fraction(ss):
    # tỉ lệ lần chạy cho ra một phép gán thỏa mọi clause (số clause không thỏa = 0)
    total = good = 0
    for rec in ss.aggregate().data(["sample", "num_occurrences"]):
        s = rec.sample
        total += int(rec.num_occurrences)
        if count_unsat(int(s["x1"]), int(s["x2"]), int(s["x3"])) == 0:
            good += int(rec.num_occurrences)
    return good / total

# Kiểm tra hàm bằng sampleset đã có ở trên (chạy được cả khi không có QPU):
print("tỉ lệ đúng của lần chạy ở Mục 3:", correct_fraction(sampleset))

if ON_QPU:
    for cs in (1, 2, 4, 8, 16):
        ss = sampler.sample(model, num_reads=200, chain_strength=cs, label=f"chain_strength={cs}")
        names = ss.record.dtype.names
        broken = float(ss.record.chain_break_fraction.mean()) if "chain_break_fraction" in names else float("nan")
        print(f"chain_strength={cs:2d}  chain đứt={broken:.3f}  tỉ lệ đúng={correct_fraction(ss):.2f}")
else:
    print("Bỏ qua phần quét chain_strength: cần QPU. Xem docs/C-hardware/dwave-leap.md để tạo tài khoản và lấy token.")

tỉ lệ đúng của lần chạy ở Mục 3: 1.0
Bỏ qua phần quét chain_strength: cần QPU. Xem docs/C-hardware/dwave-leap.md để tạo tài khoản và lấy token.
